# Real-time 0.5 T Bitter solenoid simulator (Colab / Kaggle / Jupyter)

Interactive front-end for the `bittersim` package. Uses only NumPy, SciPy, Numba, Matplotlib and ipywidgets
APIs that existed before July 2018.

**Colab:** the repository is private, so either upload the repo folder (or a zip of it) to the Colab session, or clone it with a
personal access token (`git clone https://<TOKEN>@github.com/s-siyanwal/bitter-solenoid-simulator.git`). Then run the cells below.

In [ ]:
import os, sys
# locate the repository (works when the notebook is run from notebooks/ or from a Colab clone)
for cand in ["..", "bitter-solenoid-simulator", "/content/bitter-solenoid-simulator", "/kaggle/working/bitter-solenoid-simulator"]:
    if os.path.isdir(os.path.join(cand, "package", "bittersim")):
        sys.path.insert(0, os.path.abspath(os.path.join(cand, "package")))
        break
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
from bittersim import BitterDesign, evaluate_design, fields, swissroll
from bittersim import plotting
%matplotlib inline

## 1. Real-time design explorer
The current is set automatically so that B_z(0) = B0 (E4). Every slider move re-evaluates the full electro-thermal-hydraulic model (~10 ms after JIT warm-up).

In [ ]:
KEYS = ["R1_mm", "R2_mm", "L_mm", "d_plate_mm", "D_hole_mm", "pitch_factor", "v_flow", "T_in", "dsv_mm"]
def explore(R1_mm=50.0, R2_mm=300.0, L_mm=1102.0, d_plate_mm=5.99, D_hole_mm=5.9, pitch_factor=7.95, v_flow=1.7, T_in=20.0, dsv_mm=30.0):
    d = BitterDesign(R1=R1_mm/1e3, R2=max(R2_mm, R1_mm+10)/1e3, L=L_mm/1e3, d_plate=d_plate_mm/1e3, D_hole=D_hole_mm/1e3,
                     pitch_factor=pitch_factor, v_flow=v_flow, T_in=T_in, dsv=dsv_mm/1e3)
    r = evaluate_design(d)
    flags = lambda ok: "OK " if ok else "VIOLATED"
    print("current I            %10.1f A" % r["I_A"])
    print("supply voltage       %10.3f V    [<= 8 V: %s]" % (r["V_total_V"], flags(r["V_total_V"] <= 8)))
    print("electrical power     %10.3f kW" % (r["P_elec_W"]/1e3))
    print("pump power           %10.1f W" % r["P_pump_W"])
    print("homogeneity (DSV)    %10.2f ppm  [<= 100: %s]" % (r["homogeneity_ppm"], flags(r["homogeneity_ppm"] <= 100)))
    print("hot-spot copper T    %10.2f C    [<= 85 C: %s]" % (r["T_hot_C"], flags(r["T_hot_C"] <= 85)))
    print("flow / dp / Re       %10.1f L/min  %.2f kPa  Re=%.0f" % (r["flow_L_min"], r["dp_Pa"]/1e3, r["Re"]))
    z = np.linspace(-r["L"], r["L"], 400)
    plt.figure(figsize=(7, 3)); plt.plot(z*100, fields.B_bitter_axis(r["R1"], r["R2"], r["L"], r["C_A_per_m"], z))
    plt.xlabel("z [cm]"); plt.ylabel("B_z on axis [T]"); plt.grid(alpha=.3); plt.show()

w.interact(explore,
    R1_mm=w.FloatSlider(value=50, min=50, max=100, step=0.5), R2_mm=w.FloatSlider(value=300, min=100, max=300, step=1),
    L_mm=w.FloatSlider(value=1102, min=400, max=1500, step=1), d_plate_mm=w.FloatSlider(value=5.99, min=0.5, max=6, step=0.01),
    D_hole_mm=w.FloatSlider(value=5.9, min=1, max=6, step=0.05), pitch_factor=w.FloatSlider(value=7.95, min=1.5, max=8, step=0.05),
    v_flow=w.FloatSlider(value=1.7, min=0.5, max=5, step=0.05), T_in=w.FloatSlider(value=20, min=5, max=40, step=0.5),
    dsv_mm=w.FloatSlider(value=30, min=5, max=80, step=1));

## 2. Swiss-roll RF metamaterial (acts only at RF, ~21.3 MHz)
`mu_eff(0) = 1`, so the rolls are modelled as having no effect on the static B0 field or its homogeneity. The SNR gain is an illustrative heuristic (E28).

In [ ]:
def roll(detune=1.015, loss_multiplier=50.0, N=28):
    fL = swissroll.larmor_hz(0.5)
    sr = swissroll.SwissRoll(fL*detune, N=N, loss_multiplier=loss_multiplier)
    f = np.linspace(0.97*fL, 1.03*fL, 1500); mu = sr.mu(f); m = complex(sr.mu(fL))
    print("f_L = %.4f MHz, Q = %.1f, gap = %.2f um, mu_eff(f_L) = %.3f%+.3fj, heuristic SNR gain = %.3f" % (
        fL/1e6, sr.Q, sr.gap*1e6, m.real, m.imag, float(swissroll.snr_gain(m))))
    plt.figure(figsize=(7, 3)); plt.plot(f/1e6, mu.real, label="mu'"); plt.plot(f/1e6, mu.imag, label="mu''")
    plt.axvline(fL/1e6, color="k", ls=":"); plt.legend(); plt.xlabel("f [MHz]"); plt.show()
w.interact(roll, detune=w.FloatSlider(value=1.015, min=0.95, max=1.1, step=0.0005, readout_format=".4f"),
           loss_multiplier=w.FloatSlider(value=50, min=1, max=200, step=1), N=w.IntSlider(value=28, min=5, max=60));

## 3. Constrained optimisation (differential evolution + SLSQP polish)
The README run uses `maxiter=100` with SLSQP polish (`examples/run_all.py`); the cell below uses `maxiter=20` without polish for a quicker, approximate answer.

In [ ]:
from bittersim.optimize import optimise
o = optimise(seed=1, maxiter=20, polish=False)
r = o["result"]
print(o["method"]); print({k: round(float(r[k]), 6) for k in ["R1","R2","L","d_plate","D_hole","v_flow","pitch_factor","I_A","V_total_V","P_elec_W","homogeneity_ppm","T_hot_C","dp_Pa","flow_L_min"]})

In [ ]:
plotting.fig_field(r); plotting.fig_thermal(r); plt.show()

## 4. fMRI layer: spherical harmonics, Z2/Z4 shims, B0 stability, RF SNR
Same code as `examples/run_fmri.py`. All default inputs are assumptions (see `results/fmri.json` -> `assumptions`).

In [ ]:
from bittersim import harmonics as H, stability as S, rfsnr as Q
from ipywidgets import interact, FloatSlider, Dropdown
o = evaluate_design(BitterDesign(R1=0.05, R2=0.3, L=1.102, d_plate=5.99e-3, D_hole=5.915e-3, v_flow=1.697, pitch_factor=7.962))
geo = (o["R1"], o["R2"], o["L"], o["C_A_per_m"])

def shims(dsv_mm=30.0):
    s = H.shimmed_homogeneity(*geo, dsv=dsv_mm * 1e-3)
    print("DSV %.0f mm: %.2f ppm unshimmed -> %.3f ppm with Z2/Z4 pairs; NI = %s A; %.3f W"
          % (dsv_mm, s["ppm_unshimmed"], s["ppm_shimmed"], [round(x, 2) for x in s["shim_NI_A"]], s["shim_power_W"]))
interact(shims, dsv_mm=FloatSlider(min=10, max=80, step=5, value=30))

In [ ]:
from bittersim.thermal import lumped_time_constant
from bittersim.constants import ALPHA_CU
tau = lumped_time_constant(o["P20_W"], ALPHA_CU, o["C_th_J_K"], o["R_th_K_W"])

def stab(mode="current", ripple_ppm=1.0, drift_ppm_per_h=2.0, water_amp_K=0.1, water_drift_K_per_h=0.5):
    r = S.simulate(o, tau, 0.657, dict(mode=mode, ripple_ppm=ripple_ppm, drift_ppm_per_h=drift_ppm_per_h,
                                       water_amp_K=water_amp_K, water_drift_K_per_h=water_drift_K_per_h))
    plt.figure(figsize=(7, 2.5)); plt.plot(r["t"], r["slow_ppm"]); plt.xlabel("t [s]"); plt.ylabel("dB/B slow [ppm]"); plt.show()
    print({k: round(v, 3) for k, v in r["budget_ppm"].items()}, "ppm; total %.1f Hz; meets 1 ppm: %s" % (r["budget_Hz"]["total"], r["meets_target"]))
interact(stab, mode=Dropdown(options=["current", "voltage"]), ripple_ppm=(0.0, 10.0, 0.1), drift_ppm_per_h=(0.0, 20.0, 0.5),
         water_amp_K=(0.0, 1.0, 0.01), water_drift_K_per_h=(-2.0, 2.0, 0.1))

In [ ]:
def rf(detune=1.055, loss_multiplier=50.0, t_slab_mm=50.0, depth_mm=20.0):
    c = Q.compare(B0=0.5, detune=detune, loss_multiplier=loss_multiplier, p=dict(t_slab=t_slab_mm * 1e-3, depth=depth_mm * 1e-3))
    print("mu = %.2f%+.2fj | SNR vs same coil over air %.3f | vs coil on tissue %.3f | R coil/tissue/slab = %.4f / %.4f / %.4f ohm"
          % (c["mu"][0], c["mu"][1], c["gain_vs_air"], c["gain_vs_contact"], c["slab"]["R_coil"], c["slab"]["R_tissue"], c["slab"]["R_slab"]))
interact(rf, detune=(1.001, 1.10, 0.001), loss_multiplier=(1.0, 100.0, 1.0), t_slab_mm=(5.0, 150.0, 5.0), depth_mm=(5.0, 80.0, 5.0))